In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

hourly_df = pd.read_csv('../data/processed/hourly_energy.csv')
hourly_df['timestamp'] = pd.to_datetime(hourly_df['timestamp'])
hourly_df = hourly_df.sort_values('timestamp').reset_index(drop=True)
hourly_df.head()

,timestamp,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,date,year,month,hour,day_of_week,is_weekend
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,19.0,607.0,2006-12-16,2006,12,17,Saturday,True
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0,2006-12-16,2006,12,18,Saturday,True
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0,2006-12-16,2006,12,19,Saturday,True
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0,2006-12-16,2006,12,20,Saturday,True
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0,2006-12-16,2006,12,21,Saturday,True


In [3]:
hourly_df['previous_hour_consumption'] = hourly_df['Global_active_power'].shift(1)
hourly_df['rolling_avg_3h'] = hourly_df['Global_active_power'].rolling(window=3).mean()
hourly_df['rolling_avg_24h'] = hourly_df['Global_active_power'].rolling(window=24).mean()

hourly_df = hourly_df.dropna()
hourly_df.shape

(33961, 17)

In [4]:
features = ['hour', 'month', 'is_weekend', 'previous_hour_consumption', 'rolling_avg_3h', 'rolling_avg_24h']
target = 'Global_active_power'

X = hourly_df[features]
y = hourly_df[target]

X.head()

,hour,month,is_weekend,previous_hour_consumption,rolling_avg_3h,rolling_avg_24h
23,16,12,True,2.985400,2.801356,2.499145
24,17,12,True,3.326033,3.239400,2.465140
25,18,12,True,3.406767,3.476633,2.467844
26,19,12,True,3.697100,3.337422,2.447351
27,20,12,True,2.908400,3.322333,2.451224


In [5]:
split_point = int(len(hourly_df) * 0.8)

X_train, X_test = X[:split_point], X[split_point:]
y_train, y_test = y[:split_point], y[split_point:]

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

Training rows: 27168
Testing rows: 6793


In [6]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print(f"Linear Regression Results:")
print(f"MAE: {lr_mae:.4f}")
print(f"RMSE: {lr_rmse:.4f}")
print(f"R²: {lr_r2:.4f}")

Linear Regression Results:
MAE: 0.2835
RMSE: 0.4038
R²: 0.7062


In [7]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print(f"Random Forest Results:")
print(f"MAE: {rf_mae:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R²: {rf_r2:.4f}")

Random Forest Results:
MAE: 0.2286
RMSE: 0.3475
R²: 0.7824


In [8]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(rf_model, '../models/energy_forecasting_model.pkl')
print("Model saved successfully!")

Model saved successfully!


In [9]:
import plotly.graph_objects as go

comparison_df = pd.DataFrame({
    'timestamp': hourly_df['timestamp'].iloc[split_point:].values,
    'actual': y_test.values,
    'predicted': rf_predictions
})

fig = go.Figure()
fig.add_trace(go.Scatter(x=comparison_df['timestamp'][:200], y=comparison_df['actual'][:200], name='Actual', mode='lines'))
fig.add_trace(go.Scatter(x=comparison_df['timestamp'][:200], y=comparison_df['predicted'][:200], name='Predicted', mode='lines'))
fig.update_layout(title='Actual vs Predicted Energy Consumption (first 200 test hours)')
fig.show()